# Лабораторная работа: классификация ЭКГ. Часть 1 — датасет

Лабораторная работа состоит из двух ноутбуков в двух проектах:

1. **этот ноутбук**, `ecg-warehouse/notebooks/lab_dataset.ipynb`: вы загрузите открытый датасет PTB-XL в хранилище ЭКГ `ecg-warehouse`, опишете выборку своего варианта спецификацией `DatasetSpec`, соберёте датасет и экспортируете его в проект `ecg-ml`;
2. `ecg-ml/notebooks/lab_training.ipynb`: вы обучите нейросетевой классификатор на PyTorch Lightning, оцените его на тестовой выборке и попробуете улучшить базовую модель.

[PTB-XL 1.0.3](https://physionet.org/content/ptb-xl/1.0.3/) содержит 21 799 двенадцатиканальных ЭКГ длительностью 10 с от 18 869 пациентов. Записи размечены кардиологами кодами стандарта SCP-ECG: диагностическими заключениями (`NORM`, `IMI`, `LVH`, …), заключениями о форме ЭКГ и о ритме (`SR`, `AFIB`, …).

Ноутбук выполняется в окружении проекта `ecg-warehouse`: установите его командой `uv sync` в корне репозитория и выберите ядро из `.venv`.

## Варианты

Запишите номер своего варианта в `VARIANT`, а его коды — в `CLASSES` **в том порядке, в каком они указаны в таблице**: этот порядок задаёт столбцы матрицы меток `y`, а во второй части — выходы сети и порядок классов в метриках.

| Вариант | Коды SCP-ECG (порядок классов) | Состояния |
|:-:|---|---|
| 1 | `NORM`, `AFIB` | норма, фибрилляция предсердий |
| 2 | `NORM`, `CLBBB`, `CRBBB` | норма, полная блокада левой и правой ножки пучка Гиса |
| 3 | `NORM`, `IMI`, `ASMI` | норма, нижний и переднеперегородочный инфаркт миокарда |
| 4 | `NORM`, `LVH`, `LAFB` | норма, гипертрофия левого желудочка, блокада передней ветви левой ножки |
| 5 | `NORM`, `1AVB`, `IRBBB` | норма, АВ-блокада I степени, неполная блокада правой ножки |
| 6 | `SR`, `AFIB`, `STACH`, `SBRAD` | синусовый ритм, фибрилляция предсердий, синусовая тахикардия и брадикардия |
| 7 | `NORM`, `PVC`, `AFIB` | норма, желудочковая экстрасистолия, фибрилляция предсердий |

В варианте 6 все классы — заключения о ритме, поэтому норму в нём представляет код `SR` (синусовый ритм), а не диагностическое заключение `NORM`.

Задача **многометочная** (multi-label): классы не взаимоисключающие, у одной записи может быть несколько кодов варианта сразу (например, `IMI` и `ASMI`) или ни одного.

In [1]:
import os
from pathlib import Path

# Пути в ноутбуке отсчитываются от корня репозитория, а Jupyter может запустить ноутбук из notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir("..")

### Параметры

Пути указываются относительно корня репозитория `ecg-warehouse`.

- `VARIANT` — номер вашего варианта;
- `CLASSES` — **коды вашего варианта** в порядке из таблицы, например для варианта 2: `CLASSES = ["NORM", "CLBBB", "CRBBB"]`;
- `PTB_XL_ARCHIVE` — архив с PTB-XL, `PTB_XL_PATH` — каталог, в который он распаковывается;
- `DATASET_NAME` — имя датасета в реестре хранилища;
- `DATASET_EXPORT_PATH` — куда экспортировать датасет для обучения: в каталог `data/` проекта `ecg-ml`, склонированного рядом с `ecg-warehouse`. Если вы склонировали `ecg-ml` в другое место, исправьте путь.

In [2]:
VARIANT = 1
CLASSES = ['NORM', 'AFIB']

PTB_XL_ARCHIVE = Path("data/ptb-xl.zip")
PTB_XL_PATH = Path("data/ptb-xl")

DATASET_NAME = f"ptb-xl-variant-{VARIANT}"
DATASET_EXPORT_PATH = Path("../ecg-ml/data") / DATASET_NAME

## 1. Данные

Архив [PTB-XL 1.0.3](https://physionet.org/content/ptb-xl/1.0.3/) нужно сохранить как `data/ptb-xl.zip`. Подходит архив из задания или полный ZIP с PhysioNet. Для этой работы распаковываются только метаданные и сигналы с частотой 500 Гц.

In [3]:
from tempfile import TemporaryDirectory
from zipfile import ZipFile

if not (PTB_XL_PATH / "ptbxl_database.csv").is_file():
    with ZipFile(PTB_XL_ARCHIVE) as archive:
        metadata_path, = [
            name for name in archive.namelist()
            if Path(name).name == "ptbxl_database.csv" and "__MACOSX" not in Path(name).parts
        ]
        prefix = metadata_path.removesuffix("ptbxl_database.csv")
        members = [
            name for name in archive.namelist()
            if name in {metadata_path, prefix + "scp_statements.csv"}
            or name.startswith(prefix + "records500/")
        ]
        with TemporaryDirectory(dir=PTB_XL_PATH.parent) as staging:
            archive.extractall(staging, members)
            extracted = Path(staging) / prefix
            extracted.rename(PTB_XL_PATH)


In [4]:
from ecg_warehouse.api import WarehouseAPI
from ecg_warehouse.dataset import DatasetSpec, SplitSpec

.\ecg-warehouse\data\deps\pandas\core\arrays\masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


## 2. Хранилище ЭКГ

`ecg-warehouse` хранит метаданные записей (`recordings`) и диагнозы (`diagnoses`) в таблицах Apache Iceberg, а сами сигналы — отдельно, в репозитории Icechunk. `WarehouseAPI` — точка входа для загрузки данных, SQL-запросов и сборки датасетов.

Конфигурация `cfg/local.json` описывает локальное хранилище в каталоге `data/`: каталог таблиц Iceberg — `data/catalog.db`, сигналы — `data/ECG/signals.icechunk`.

In [5]:
wh = WarehouseAPI("cfg/local.json")

### Загрузка PTB-XL

`ingest_ptb_xl` читает записи с частотой дискретизации 500 Гц (12 отведений × 5000 отсчётов) и их коды SCP-ECG. Загрузка всего датасета занимает заметное время. Если её прервать, повторный запуск пропустит уже загруженные записи.

In [6]:
wh.ingest_ptb_xl(data_dir=PTB_XL_PATH, rate=500)

  0%|                                                                     | 0/21799 [00:00<?, ?it/s]

  8%|████▍                                                  | 1759/21799 [00:00<00:01, 17574.18it/s]

 18%|█████████▊                                             | 3895/21799 [00:00<00:00, 19796.21it/s]

 29%|███████████████▊                                       | 6266/21799 [00:00<00:00, 21580.73it/s]

 39%|█████████████████████▌                                 | 8557/21799 [00:00<00:00, 22102.90it/s]

 50%|███████████████████████████▏                          | 10997/21799 [00:00<00:00, 22930.22it/s]

... (2 outputs left out)

 84%|█████████████████████████████████████████████▍        | 18335/21799 [00:00<00:00, 23968.08it/s]

 95%|███████████████████████████████████████████████████▌  | 20804/21799 [00:00<00:00, 24192.46it/s]

100%|██████████████████████████████████████████████████████| 21799/21799 [00:00<00:00, 23039.78it/s]

Signals store cleanup: freed 0.0 MiB


`wh.sql` выполняет SQL-запрос (DuckDB) к таблицам `recordings` и `diagnoses` и возвращает `pandas.DataFrame`. Проверим, сколько записей и пациентов загружено.

In [7]:
wh.sql(
    "SELECT source, sampling_rate_hz, num_leads, count(*) AS recordings, count(DISTINCT patient_id) AS patients FROM recordings GROUP BY ALL"
)

,source,sampling_rate_hz,num_leads,recordings,patients
0,ptb-xl-1.0.3,500,12,21799,18869


### Баланс классов

Считаю уникальные записи каждого класса среди 12-канальных ЭКГ длительностью 10 с. Одна запись может учитываться в нескольких классах, поэтому доли рассчитаны от всех записей выбранного варианта и могут в сумме превышать 100%.

In [8]:
from IPython.display import display

codes = ", ".join("'" + code.replace("'", "''") + "'" for code in CLASSES)
selected = wh.sql(f"""
    SELECT d.recording_id, d.code
    FROM diagnoses AS d
    JOIN recordings AS r USING (recording_id)
    WHERE d.code_system = 'SCP-ECG' AND d.code IN ({codes})
      AND r.source = 'ptb-xl-1.0.3' AND r.sampling_rate_hz = 500
      AND r.num_leads = 12 AND r.duration_sec = 10
    GROUP BY ALL
""")
counts = selected.groupby("code")["recording_id"].nunique().reindex(CLASSES, fill_value=0)
n_selected = selected["recording_id"].nunique()
assert n_selected and (counts > 0).all()
display(counts.to_frame("Записей").assign(
    Доля_процентов=(100 * counts / n_selected).round(2)
))
print(f"Всего записей: {n_selected}; с несколькими метками: "
      f"{(selected.groupby('recording_id').size() > 1).sum()}.")
print(f"Чаще всего встречается {counts.idxmax()}; реже всего — {counts.idxmin()}. "
      f"Отношение частот: {counts.max() / counts.min():.2f}.")


,Записей,Доля_процентов
code,,
NORM,9514,86.56
AFIB,1514,13.77


Всего записей: 10991; с несколькими метками: 37.
Чаще всего встречается NORM; реже всего — AFIB. Отношение частот: 6.28.


## 3. Сборка датасета

`DatasetSpec` декларативно описывает выборку: по спецификации и текущему состоянию таблиц датасет собирается воспроизводимо.

- `name` — имя датасета в реестре хранилища;
- `sources`, `num_leads`, `sampling_rate_hz`, `min_duration_sec`, `max_duration_sec` — фильтры записей: только PTB-XL, 12 отведений, 500 Гц, ровно 10 с;
- `code_systems`, `codes` — оставить только диагнозы SCP-ECG с кодами из `CLASSES`;
- `recording_columns`, `diagnosis_columns` — столбцы метаданных, которые попадут в датасет;
- `split` — разбиение на train и test **по пациентам**: все записи пациента попадают в одну часть, в test уходит около 20 % пациентов, `seed` фиксирует разбиение. Так записи одного человека не окажутся одновременно в обучении и в тесте.
- `require_diagnosis=True` — экспортировать записи с хотя бы одной меткой варианта.

In [9]:
dataset_spec = DatasetSpec(
    name=DATASET_NAME,
    sources=["ptb-xl-1.0.3"],
    num_leads=12,
    sampling_rate_hz=500,
    code_systems=["SCP-ECG"],
    codes=CLASSES,
    require_diagnosis=True,
    min_duration_sec=10,
    max_duration_sec=10,
    recording_columns=[
        "recording_id",
        "source",
        "patient_id",
        "sampling_rate_hz",
        "duration_sec",
        "age_at_recording",
        "sex",
    ],
    diagnosis_columns=[
        "diagnosis_id",
        "recording_id",
        "code",
        "code_system",
        "label_text",
    ],
    split=SplitSpec(test_fraction=0.2, seed=0),
)

`build_dataset` собирает датасет по спецификации и регистрирует его в реестре хранилища под именем и версией. Ячейка выводит манифест сборки: имя, версию, спецификацию и описание файлов. Сигналы не копируются — датасет ссылается на них в хранилище.

In [10]:
dataset = wh.build_dataset(spec=dataset_spec)
assert dataset["artifacts"]["recordings"]["rows"] == n_selected
dataset

Built dataset ptb-xl-variant-1@0.0.0-dev.hf2fbdb14dfa85e94 (dev): 10991 recordings, 11028 diagnoses


{'name': 'ptb-xl-variant-1',
 'version': '0.0.0-dev.hf2fbdb14dfa85e94',
 'mode': 'dev',
 'release_type': None,
 'content_hash': 'f2fbdb14dfa85e94',
 'spec': {'name': 'ptb-xl-variant-1',
  'sources': ['ptb-xl-1.0.3'],
  'num_leads': 12,
  'sampling_rate_hz': 500,
  'min_duration_sec': 10.0,
  'max_duration_sec': 10.0,
  'code_systems': ['SCP-ECG'],
  'codes': ['NORM', 'AFIB'],
  'drop_recordings_with_other_codes': False,
  'require_diagnosis': True,
  'recording_columns': ['recording_id',
   'source',
   'patient_id',
   'sampling_rate_hz',
   'duration_sec',
   'age_at_recording',
   'sex'],
  'diagnosis_columns': ['diagnosis_id',
   'recording_id',
   'code',
   'code_system',
   'label_text'],
  'signals_format': 'icechunk',
  'split': {'test_fraction': 0.2, 'seed': 0}},
 'snapshots': {'recordings': 6799528071134462185,
  'diagnoses': 1710770629004569519},
 'artifacts': {'recordings': {'uri': 'file://G:\\nauchka\\utilities\\project\\ecg-labwork\\ecg-warehouse\\data\\datasets\\ptb-xl-

Экспорт в формат `npz` записывает в `DATASET_EXPORT_PATH`:

- `signals.npz` с массивами `X` — сигналы `float32` формы `(записи, 12, 5000)`; `y` — метки формы `(записи, len(CLASSES))`, где `y[i, j] = 1`, если у записи `i` есть код `CLASSES[j]`; а также `classes`, `recording_id` и `split` (`train` / `test`);
- `meta.csv` — метаданные записей, `diagnoses.csv` — их диагнозы.

In [11]:
wh.export_dataset(
    dataset_ref=f"{dataset['name']}@{dataset['version']}",
    format="npz",
    output=DATASET_EXPORT_PATH,
)

  0%|                                                                     | 0/10991 [00:00<?, ?it/s]

  0%|                                                           | 1/10991 [00:01<4:13:05,  1.38s/it]

  4%|██▌                                                       | 492/10991 [00:01<00:22, 460.67it/s]

  7%|████                                                      | 781/10991 [00:02<00:30, 331.79it/s]

  9%|█████▎                                                   | 1025/10991 [00:04<00:39, 252.67it/s]

 14%|███████▊                                                 | 1516/10991 [00:04<00:19, 485.06it/s]

... (26 outputs left out)

 97%|██████████████████████████████████████████████████████▎ | 10654/10991 [00:25<00:00, 582.65it/s]

 99%|███████████████████████████████████████████████████████▎| 10860/10991 [00:25<00:00, 540.26it/s]

100%|████████████████████████████████████████████████████████| 10991/10991 [00:25<00:00, 427.49it/s]

{'name': 'ptb-xl-variant-1',
 'version': '0.0.0-dev.hf2fbdb14dfa85e94',
 'format': 'npz',
 'output': 'G:\\nauchka\\utilities\\project\\ecg-labwork\\ecg-ml\\data\\ptb-xl-variant-1',
 'subset': {'split': None, 'limit': None},
 'recordings': {'dataset': 10991, 'exported': 10991},
 'diagnoses': {'dataset': 11028, 'exported': 11028},
 'files': {'signals.npz': {'X': [10991, 12, 5000],
   'y': [10991, 2],
   'classes': ['NORM', 'AFIB'],
   'recording_id': [10991],
   'split': [10991]},
  'meta.csv': {'rows': 10991},
  'diagnoses.csv': {'rows': 11028}}}

In [12]:
import numpy as np
import pandas as pd

meta = pd.read_csv(DATASET_EXPORT_PATH / "meta.csv", dtype={"patient_id": str})
assert meta.groupby(["source", "patient_id"])["split"].nunique().max() == 1
assert len(meta) == n_selected
with np.load(DATASET_EXPORT_PATH / "signals.npz") as export:
    assert export["classes"].tolist() == CLASSES
    labels = export["y"]
    assert labels.shape == (len(meta), len(CLASSES))
    assert labels.any(axis=1).all()
    assert (labels.sum(axis=0) == counts.to_numpy()).all()
    distribution = pd.DataFrame(labels, columns=CLASSES).groupby(export["split"]).sum()
    distribution.insert(0, "ЭКГ", pd.Series(export["split"]).value_counts())
display(distribution)
print("Пересечений по пациентам между train и test нет.")


,ЭКГ,NORM,AFIB
test,2240,1911,335
train,8751,7603,1179


Пересечений по пациентам между train и test нет.


## Что дальше

1. Сохраните ноутбук вместе с результатами ячеек и закоммитьте его в свою ветку, как описано в инструкции к лабораторной работе.
2. Переходите ко второй части: `ecg-ml/notebooks/lab_training.ipynb`.